In [1]:
import os
import certifi
import requests

from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub

from langchain_classic.agents import create_react_agent, AgentExecutor

from langchain_community.llms import Ollama
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langchain.tools import tool

C:\Users\Revanth L\AppData\Local\Temp\ipykernel_33516\1780524494.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [14]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

# OPEN_API_KEY = os.getenv("OPEN_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [3]:
llm = Ollama(
    model="qwen2.5:3b",
    temperature=0
)

# base_llm = HuggingFaceEndpoint(
#     repo_id="meta-llama/Llama-3.1-8B-Instruct",
#     task="text-generation"
# )

# llm = ChatHuggingFace(llm=base_llm)

C:\Users\Revanth L\AppData\Local\Temp\ipykernel_33516\1289999859.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [16]:

import os
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool

# Load API key from .env
load_dotenv(override=True)

WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    if not WEATHERSTACK_API_KEY:
        return "Weatherstack API key is missing."

    url = "https://api.weatherstack.com/current"

    params = {
        "access_key": WEATHERSTACK_API_KEY,
        "query": city,
        "units": "m"
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

    except requests.RequestException as e:
        return f"Weather API request failed: {e}"

    if "current" not in data:
        return (
            f"Could not fetch weather data: "
            f"{data.get('error', {}).get('info', 'Unknown error')}"
        )

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )


In [19]:
result = get_weather_data.invoke({"city": "Bangalore"})
print(result)

City: Bangalore
Temperature: 28°C
Weather: Cloudy 
Humidity: 34%


In [22]:
search_tool = TavilySearchResults(max_results=2)
search_tool.invoke("What is Tavily")

C:\Users\Revanth L\AppData\Local\Temp\ipykernel_33516\4024580342.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


[{'title': 'What is Tavily? · Research Tool Guide',
  'url': 'https://buildrlabs.ai/tool-guide/tavily',
  'content': "Research Freemium\n\n Visit Tavily\n\n## What is Tavily?\n\nTavily is a search API built specifically for AI agents and retrieval-augmented generation, known as RAG, a technique where an LLM pulls in outside information before answering. Instead of returning a list of links for a human to click, Tavily returns reranked, information-dense snippets that an LLM can reason over directly. It also handles extracting clean content from a URL, crawling a whole site, and generating a synthesised research report from a single query. For developers building agents that need current, factual information, Tavily removes the work of scraping and cleaning web data yourself.\n\n## How Tavily works [...] ## FAQ\n\nWhat is Tavily used for? + \n\nTavily gives AI agents and RAG applications a way to search, extract, and crawl the live web. Developers use it to answer questions an LLM canno

In [23]:
from langchain_core.prompts import PromptTemplate

template = """
Answer the following questions as best you can. You have access to these tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template='\nAnswer the following questions as best you can. You have access to these tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat)\nThought: I now know the final answer\nFinal Answer: the final answer to the original question\n\nBegin!\n\nQuestion: {input}\nThought: {agent_scratchpad}\n')

In [24]:
tools = [search_tool, get_weather_data]

In [25]:
# ReAct : It is design pattern used in AI agents that stands for Reasoning + Acting. It allows LLMs to interleave internal reasoning(Thought) and external actions(like tool use) in a structured, multi-step process.

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [26]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True   
)

In [27]:
response = agent_executor.invoke({
    "input": (
        "I need capital of india and its weather"
    )
})



> Entering new AgentExecutor chain...
 I need to find the capital of India and also fetch the current weather information for India. I will first search for the capital of India and then fetch the weather data.
Action: tavily_search_results_json
Action Input: capital of India[{'title': 'New Delhi | History, Population, Map, & Facts | Britannica', 'url': 'https://www.britannica.com/place/New-Delhi', 'content': "As one of India’s largest urban areas, Delhi is a commercial, transport, and cultural hub, as well as the country’s political center. The city’s cultural life displays a mix of traditional and cosmopolitan styles. Numerous museums, historical monuments, libraries, and gardens can be found throughout the city.\n\n### Where is Delhi located in India?\n\nDelhi, a city and union territory, is the national capital of India, located in the north-central part of the country. It consists of two main parts: Old Delhi, the historic city in the north, and New Delhi, the capital of India s

In [88]:
print(response["output"])

The current population of India is approximately 1.414 billion people. The UN's projections suggest that the population will reach 1.7 billion by 2050. The current weather data for India will be fetched using the get_weather_data tool.
